In [ ]:
!pip install faiss-cpu

In [ ]:
#ran on t4 gpu on collab
import zipfile
import json
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import faiss
import time

In [ ]:
with zipfile.ZipFile("aapd.zip") as z:
    with z.open("aapd.json") as f:
        aapd = json.load(f)

print(type(aapd))
print((aapd.keys()))
print(aapd["data"].keys())

<class 'dict'>
dict_keys(['meta', 'label_set', 'data'])
dict_keys(['train', 'val', 'test'])


In [ ]:
aapd_df_train = pd.DataFrame(aapd["data"]["train"])
aapd_df_val = pd.DataFrame(aapd["data"]["val"])
aapd_df_test = pd.DataFrame(aapd["data"]["test"])

In [ ]:
aapd_df_train["text"].str.split().str.len().describe()


,text
count,53840.000000
mean,163.452006
std,67.623457
min,1.000000
25%,114.000000
50%,157.500000
75%,208.000000
max,504.000000


In [ ]:
aapd_df_val["text"].str.split().str.len().describe()

,text
count,1000.000000
mean,162.120000
std,67.669186
min,7.000000
25%,114.000000
50%,155.000000
75%,206.000000
max,522.000000


In [ ]:
print(aapd_df_train.isna().sum())
print(aapd_df_val.isna().sum())
print(aapd_df_test.isna().sum())
#no missing values anywhere

id        0
text      0
labels    0
dtype: int64
id        0
text      0
labels    0
dtype: int64
id        0
text      0
labels    0
dtype: int64


In [ ]:
#greedy selection - optimising to have as much unique labels as possible in the top_k (within the 20 most similiar returned)
def greedy_label_selection(pool_indices, pool_distances, train_df, target_k=5):
    selected_indices = []
    selected_scores = []
    current_covered_labels = set()

    #start with the #1 most similar result to maintain relevance
    first_idx = pool_indices[0]
    selected_indices.append(int(first_idx))
    selected_scores.append(float(pool_distances[0]))
    current_covered_labels.update(train_df.iloc[first_idx]["labels"])

    #iteratively pick from the remaining pool
    remaining_pool_indices = list(range(1, len(pool_indices)))

    while len(selected_indices) < target_k and remaining_pool_indices:
        best_candidate_pool_pos = -1
        max_new_labels = -1

        for pos in remaining_pool_indices:
            candidate_train_idx = pool_indices[pos]
            candidate_labels = set(train_df.iloc[candidate_train_idx]["labels"])

            #count how many labels this candidate adds that are not there yet
            new_labels = len(candidate_labels - current_covered_labels)

            #selection criteria: most new labels
            #if tie-break - highest similarity
            if new_labels > max_new_labels:
                max_new_labels = new_labels
                best_candidate_pool_pos = pos

        #add the best candidate found in this pass
        if best_candidate_pool_pos != -1:
            chosen_train_idx = pool_indices[best_candidate_pool_pos]
            selected_indices.append(int(chosen_train_idx))
            selected_scores.append(float(pool_distances[best_candidate_pool_pos]))
            current_covered_labels.update(train_df.iloc[chosen_train_idx]["labels"])
            remaining_pool_indices.remove(best_candidate_pool_pos)
        else:
            break

    return selected_indices, selected_scores

In [ ]:
#standard top-k selection (will test which strategy is better when prompting)
def standard_top_k_selection(pool_indices, pool_distances, target_k=5):
    selected_indices = [int(idx) for idx in pool_indices[:target_k]]
    selected_scores = [float(score) for score in pool_distances[:target_k]]

    return selected_indices, selected_scores

In [ ]:
MODEL_NAME = "BAAI/bge-m3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def sync_gpu():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

experiment_start = time.perf_counter()

#model initialization
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
model.max_seq_length = 1024 #increased from 512, bcs the longest text in train set has over 500 words (so even more tokens)

aapd_df_train = aapd_df_train.reset_index(drop=True)
aapd_df_test = aapd_df_test.reset_index(drop=True)
aapd_df_val = aapd_df_val.reset_index(drop=True)

train_texts = aapd_df_train["text"].tolist()
test_texts = aapd_df_test["text"].tolist()
val_texts = aapd_df_val["text"].tolist()

#encode training set
sync_gpu()
start = time.perf_counter()

train_embeddings = model.encode(
    train_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,).astype("float32")

sync_gpu()
train_encoding_time = time.perf_counter() - start


#index
start = time.perf_counter()

dimension = train_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(train_embeddings)

index_build_time = time.perf_counter() - start

np.save("aapd_train_bge_m3_embeddings.npy", train_embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/1683 [00:00<?, ?it/s]

In [ ]:
POOL_SIZE = 20  #large pool to pick from
TARGET_K = 5    #final number of examples

#encode only once  (since it is the same, regardless of strategy)
def encode_and_search(query_df):
    query_texts = query_df["text"].tolist()

    sync_gpu()
    start = time.perf_counter()

    query_embeddings = model.encode(query_texts, batch_size=32, show_progress_bar=True,normalize_embeddings=True).astype("float32")

    sync_gpu()
    encoding_time = time.perf_counter() - start

    sync_gpu()
    start_search = time.perf_counter()

    distances_pool, indices_pool = index.search(query_embeddings, POOL_SIZE)

    sync_gpu()
    search_time = time.perf_counter() - start_search

    return distances_pool, indices_pool, encoding_time, search_time

In [ ]:
def build_retrieval_results(query_df, distances_pool, indices_pool, strategy):
    retrieval_results = {}

    start_selection = time.perf_counter()

    for i in range(len(indices_pool)):
        if strategy == "greedy":
            sel_indices, sel_scores = greedy_label_selection(
                indices_pool[i],
                distances_pool[i],
                aapd_df_train,
                target_k=TARGET_K)

        elif strategy == "top_k":
            sel_indices, sel_scores = standard_top_k_selection(
                indices_pool[i],
                distances_pool[i],
                target_k=TARGET_K)

        else:
            raise ValueError("Unknown strategy.")

        sel_indices = [int(idx) for idx in sel_indices]
        sel_scores = [float(score) for score in sel_scores]

  #creating a json which contains not only the indexes and similarity, but also the label names and texts
  #for audit only
        retrieval_results[i] = {
            "query_labels": query_df.iloc[i]["labels"],
            "retrieved_train_indices": sel_indices,
            "similarity_scores": sel_scores,
            "retrieved_train_labels": [
                aapd_df_train.iloc[idx]["labels"] for idx in sel_indices],
            "retrieved_train_texts": [
                aapd_df_train.iloc[idx]["text"] for idx in sel_indices]}

    selection_time = time.perf_counter() - start_selection

    return retrieval_results, selection_time

In [ ]:
#val
val_distances_pool, val_indices_pool, val_encoding_time, val_search_time = encode_and_search(aapd_df_val)
val_retrieval_results_greedy, val_selection_time_greedy = build_retrieval_results(query_df=aapd_df_val, distances_pool=val_distances_pool, indices_pool=val_indices_pool, strategy="greedy")
val_retrieval_results_topk, val_selection_time_topk = build_retrieval_results(query_df=aapd_df_val,distances_pool=val_distances_pool, indices_pool=val_indices_pool, strategy="top_k")

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:
#test
test_distances_pool, test_indices_pool, test_encoding_time, test_search_time = encode_and_search(aapd_df_test)
test_retrieval_results_greedy, test_selection_time_greedy = build_retrieval_results(query_df=aapd_df_test, distances_pool=test_distances_pool, indices_pool=test_indices_pool, strategy="greedy")
test_retrieval_results_topk, test_selection_time_topk = build_retrieval_results(query_df=aapd_df_test, distances_pool=test_distances_pool, indices_pool=test_indices_pool, strategy="top_k")

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:
retrieval_files = {
    "aapd_validation_greedy_retrieval_results.json": val_retrieval_results_greedy,
    "aapd_validation_top_k_retrieval_results.json": val_retrieval_results_topk,
    "aapd_test_greedy_retrieval_results.json": test_retrieval_results_greedy,
    "aapd_test_top_k_retrieval_results.json": test_retrieval_results_topk}

for output_path, retrieval_results in retrieval_files.items():
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(retrieval_results, f, ensure_ascii=False, indent=2)

In [ ]:
timing_results = {
    "model_name": MODEL_NAME,
    "dataset": "AAPD",
    "device": DEVICE,
    "max_seq_length": 1024,
    "batch_size": 32,
    "pool_size": POOL_SIZE,
    "n_retrieved": TARGET_K,
    "train_encoding_time_sec": train_encoding_time,
    "index_build_time_sec": index_build_time,
    "validation_encoding_time_sec": val_encoding_time,
    "validation_faiss_search_time_sec": val_search_time,
    "validation_greedy_selection_time_sec": val_selection_time_greedy,
    "validation_topk_selection_time_sec": val_selection_time_topk,
    "validation_faiss_ms_per_query": (val_search_time / len(aapd_df_val)) * 1000,
    "test_encoding_time_sec": test_encoding_time,
    "test_faiss_search_time_sec": test_search_time,
    "test_greedy_selection_time_sec": test_selection_time_greedy,
    "test_topk_selection_time_sec": test_selection_time_topk,
    "test_faiss_ms_per_query": (test_search_time / len(aapd_df_test)) * 1000,
    "total_experiment_time_sec": time.perf_counter() - experiment_start}

with open("aapd_retrieval_timing_results.json", "w", encoding="utf-8") as f:
    json.dump(timing_results, f, indent=2)

print("\nTiming results:")
for key, value in timing_results.items():
    print(f"{key}: {value}")


Timing results:
model_name: BAAI/bge-m3
dataset: AAPD
device: cuda
max_seq_length: 1024
batch_size: 32
pool_size: 20
n_retrieved: 5
train_encoding_time_sec: 3244.089267077
index_build_time_sec: 0.17662446700023793
validation_encoding_time_sec: 60.71732023200002
validation_faiss_search_time_sec: 1.0295962889999828
validation_greedy_selection_time_sec: 1.5212020099997972
validation_topk_selection_time_sec: 0.18332925799995792
validation_faiss_ms_per_query: 1.0295962889999828
test_encoding_time_sec: 62.560379521999494
test_faiss_search_time_sec: 1.0245536069996888
test_greedy_selection_time_sec: 2.415018812999733
test_topk_selection_time_sec: 0.3211860979999983
test_faiss_ms_per_query: 1.0245536069996888
total_experiment_time_sec: 3433.917718172


In [ ]:
#sanity check
#do the retreived samples actually contain the labels of the test set?
#calculating coverage metric (the number of queries for which all labels are covered across the retreived texts)

def run_label_audit(json_path, dataset_name):
    with open(json_path, "r", encoding="utf-8") as f:
        results = json.load(f)

    full_coverage_count = 0
    recall_scores = []
    similarity_scores = []

    for query_id, data in results.items():
        target_labels = set(data["query_labels"])
        #flatten the list-of-lists containing the labels from the top5k neighbours
        pooled_candidate_labels = set(label for neighbor_labels in data["retrieved_train_labels"] for label in neighbor_labels)

        #Metric 1: Full Coverage (all target labels found in the top-k)
        if target_labels.issubset(pooled_candidate_labels):
            full_coverage_count += 1

        #Metric 2: proportion of gold labels covered by the retrieved examples (e.g., the query had 3 labels, but only 1 of them was present in the retrieved set)
        intersected = target_labels.intersection(pooled_candidate_labels)
        recall_scores.append(len(intersected) / len(target_labels))

        #Metric 3: Cosine Similarity of the retreived examples
        similarity_scores.extend(data["similarity_scores"])

    total_queries = len(results)
    stats = {
        "Dataset": dataset_name,
        "Number of Queries": total_queries,
        "Full Label Coverage@5": f"{(full_coverage_count / total_queries) * 100:.2f}%",
        "Mean Label Recall @5": f"{np.mean(recall_scores) * 100:.2f}%",
        "Mean Cosine Similarity @5": f"{np.mean(similarity_scores):.2f}"}

    return stats

val_stats_greedy = run_label_audit("aapd_validation_greedy_retrieval_results.json", "Validation Greedy")
val_stats_topk = run_label_audit("aapd_validation_top_k_retrieval_results.json", "Validation Top-K")
test_stats_greedy = run_label_audit("aapd_test_greedy_retrieval_results.json", "Test Greedy")
test_stats_topk = run_label_audit("aapd_test_top_k_retrieval_results.json", "Test Top-K")

print("Validation sets")
print(json.dumps(val_stats_greedy, indent=2))
print(json.dumps(val_stats_topk, indent=2))

print("Test sets")
print(json.dumps(test_stats_greedy, indent=2))
print(json.dumps(test_stats_topk, indent=2))

Validation sets
{
  "Dataset": "Validation Greedy",
  "Number of Queries": 1000,
  "Full Label Coverage@5": "87.10%",
  "Mean Label Recall @5": "94.35%",
  "Mean Cosine Similarity @5": "0.70"
}
{
  "Dataset": "Validation Top-K",
  "Number of Queries": 1000,
  "Full Label Coverage@5": "80.00%",
  "Mean Label Recall @5": "90.34%",
  "Mean Cosine Similarity @5": "0.71"
}
Test sets
{
  "Dataset": "Test Greedy",
  "Number of Queries": 1000,
  "Full Label Coverage@5": "84.10%",
  "Mean Label Recall @5": "93.05%",
  "Mean Cosine Similarity @5": "0.69"
}
{
  "Dataset": "Test Top-K",
  "Number of Queries": 1000,
  "Full Label Coverage@5": "75.10%",
  "Mean Label Recall @5": "87.87%",
  "Mean Cosine Similarity @5": "0.71"
}


In [ ]:
# creating input files which will be used by the LLM (not containing query labels)
def create_llm_data_file(query_df, retrieval_results, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for i, item in retrieval_results.items():
            i = int(i)

            retrieved_examples = []

            for idx,  labels, text in zip(
                item["retrieved_train_indices"],
                item["retrieved_train_labels"],
                item["retrieved_train_texts"]):
                retrieved_examples.append({
                    "train_index": int(idx),
                    "labels": labels,
                    "text": text})

            row = {
                "query_id": i,
                "target_text": query_df.iloc[i]["text"],
                "retrieved_examples": retrieved_examples}

            f.write(json.dumps(row, ensure_ascii=False) + "\n")

# validation set files
create_llm_data_file(query_df=aapd_df_val, retrieval_results=val_retrieval_results_greedy,output_path="aapd_validation_greedy_llm_input_data.jsonl")
create_llm_data_file(query_df=aapd_df_val, retrieval_results=val_retrieval_results_topk, output_path="aapd_validation_top_k_llm_input_data.jsonl")

# test set files
create_llm_data_file(query_df=aapd_df_test, retrieval_results=test_retrieval_results_greedy, output_path="aapd_test_greedy_llm_input_data.jsonl")
create_llm_data_file(query_df=aapd_df_test, retrieval_results=test_retrieval_results_topk, output_path="aapd_test_top_k_llm_input_data.jsonl")

In [ ]:
#files for later LLM evaluation (with gold labels)
#but this is optional (might delete later), since mlb can be used for classification report
def create_gold_label_file(query_df, output_path):
    gold_labels = {}

    for i in range(len(query_df)):
        gold_labels[int(i)] = query_df.iloc[i]["labels"]

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(gold_labels, f, ensure_ascii=False, indent=2)

In [ ]:
create_gold_label_file(query_df=aapd_df_val, output_path="aapd_validation_gold_labels.json")
create_gold_label_file(query_df=aapd_df_test, output_path="aapd_test_gold_labels.json")